In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Clasificación: una Qiskit Function de Multiverse Computing
> **Note:** * Las Qiskit Functions son una función experimental disponible únicamente para usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y pueden cambiar.

## Descripción general
Con la función "Singularity Machine Learning - Clasificación" puedes resolver problemas reales de aprendizaje automático en hardware cuántico sin necesitar experiencia en computación cuántica. Esta función de aplicación, basada en métodos ensemble, es un clasificador híbrido. Aprovecha métodos clásicos como boosting, bagging y stacking para el entrenamiento inicial del ensemble. Posteriormente, se emplean algoritmos cuánticos como el solucionador cuántico variacional de valores propios (VQE) y el algoritmo cuántico de optimización aproximada (QAOA) para mejorar la diversidad, las capacidades de generalización y la complejidad global del ensemble entrenado.

A diferencia de otras soluciones de aprendizaje automático cuántico, esta función es capaz de manejar conjuntos de datos a gran escala con millones de ejemplos y características sin estar limitada por el número de qubits del QPU objetivo. El número de qubits solo determina el tamaño del ensemble que se puede entrenar. También es altamente flexible y puede utilizarse para resolver problemas de clasificación en una amplia variedad de dominios, incluyendo finanzas, salud y ciberseguridad.
Logra de manera consistente altas precisiones en problemas clásicamente difíciles que involucran conjuntos de datos de alta dimensión, ruidosos y desbalanceados.
![Cómo funciona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Está diseñada para:
1. Ingenieros y científicos de datos en empresas que buscan mejorar sus ofertas tecnológicas integrando el aprendizaje automático cuántico en sus productos y servicios,
2. Investigadores en laboratorios de investigación cuántica que exploran aplicaciones de aprendizaje automático cuántico y buscan aprovechar la computación cuántica para tareas de clasificación, y
3. Estudiantes y docentes en instituciones educativas en cursos como aprendizaje automático, que buscan demostrar las ventajas de la computación cuántica.

El siguiente ejemplo muestra sus diversas funcionalidades, incluyendo `create`, `list`, `fit` y `predict`, y demuestra su uso en un problema sintético compuesto por dos semiciclos entrelazados, un problema notoriamente difícil debido a su frontera de decisión no lineal.


## Descripción de la función
Esta Qiskit Function permite a los usuarios resolver problemas de clasificación binaria utilizando el clasificador ensemble mejorado cuánticamente de Singularity. En segundo plano, utiliza un enfoque híbrido para entrenar clásicamente un ensemble de clasificadores sobre el conjunto de datos etiquetado y, a continuación, optimizarlo para maximizar la diversidad y la generalización mediante el Algoritmo Cuántico de Optimización Aproximada (QAOA) en QPUs de IBM&reg;. A través de una interfaz fácil de usar, los usuarios pueden configurar un clasificador según sus requisitos, entrenarlo con el conjunto de datos de su elección y utilizarlo para hacer predicciones sobre un conjunto de datos no visto previamente.

Para resolver un problema de clasificación genérico:
1. Preprocesa el conjunto de datos y divídelo en conjuntos de entrenamiento y prueba. Opcionalmente, puedes dividir además el conjunto de entrenamiento en conjuntos de entrenamiento y validación. Esto se puede lograr usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Si el conjunto de entrenamiento está desbalanceado, puedes remuestrearlo para equilibrar las clases usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Sube los conjuntos de entrenamiento, validación y prueba por separado al almacenamiento de la función usando el método `file_upload` del catálogo, pasándole la ruta correspondiente cada vez.
4. Inicializa el clasificador cuántico usando la acción `create` de la función, que acepta hiperparámetros como el número y los tipos de aprendices, la regularización (valor lambda) y las opciones de optimización incluyendo el número de capas, el tipo de optimizador clásico, el backend cuántico, entre otros.
5. Entrena el clasificador cuántico sobre el conjunto de entrenamiento usando la acción `fit` de la función, pasándole el conjunto de entrenamiento etiquetado y el conjunto de validación si corresponde.
6. Realiza predicciones sobre el conjunto de prueba no visto previamente usando la acción `predict` de la función.
## Enfoque basado en acciones
La función utiliza un enfoque basado en acciones. Puedes pensarlo como un entorno virtual donde usas acciones para realizar tareas o cambiar su estado. Actualmente, ofrece las siguientes acciones: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) y [create_fit_predict](#7-create-fit-predict). El siguiente ejemplo demuestra la acción `create_fit_predict`.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List
La acción `list` recupera todos los clasificadores almacenados en formato `*.pkl.tar` del directorio de datos compartido. También puedes acceder al contenido de este directorio usando el método `catalog.files()`. En general, la acción list busca archivos con la extensión `*.pkl.tar` en el directorio de datos compartido y los devuelve en formato de lista.
#### Entradas
|     Nombre    |    Tipo     | Descripción |   Requerido  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | El nombre de la acción entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` y `delete`. | Sí |

#### Uso

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create
La acción `create` crea un clasificador del tipo `quantum_classifier` especificado usando los parámetros proporcionados, y lo guarda en el directorio de datos compartido.

> **Note:** La función actualmente solo admite `QuantumEnhancedEnsembleClassifier`.
#### Entradas
|     Nombre    |    Tipo     | Descripción | Requerido  | Por defecto |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | El nombre de la acción entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` y `delete`. | Sí | - |
| `name` | `str` | El nombre del clasificador cuántico, p. ej., `spam_classifier`. | Sí | - |
| `instance` | `str` | Instancia de IBM. | Sí | - |
| `backend_name` | `str` | Recurso de cómputo de IBM. El valor por defecto es `None`, lo que significa que se usará el backend con el menor número de trabajos pendientes. | No | `None` |
| `quantum_classifier` | `str` | El tipo de clasificador cuántico, es decir, `QuantumEnhancedEnsembleClassifier`. | No | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | El número de aprendices en el ensemble. | No | `10` |
| `learners_types` | `list` | Tipos de aprendices. Entre los tipos admitidos se encuentran: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` y `LogisticRegression`. Más detalles sobre cada uno se pueden encontrar en la [documentación de scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | No | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporciones de cada tipo de aprendiz en el ensemble. | No | `[1.0]` |
| `learners_options` | `list` | Opciones para cada tipo de aprendiz en el ensemble. Para una lista completa de opciones correspondientes al tipo o tipos de aprendiz elegido, consulta la [documentación de scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | No | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` o `list` | Tipo(s) de regularización a utilizar: `onsite` o `alpha`. `onsite` controla el término onsite donde valores más altos llevan a ensembles más dispersos. `alpha` controla el equilibrio entre los términos de interacción y onsite donde valores más bajos llevan a ensembles más dispersos. Si se proporciona una lista, se entrenarán modelos para cada tipo y se seleccionará el de mejor rendimiento. | No | `onsite` |
| `regularization` | `str` o `float` o `list` | Valor de regularización. Acotado entre `0` y `+inf` si `regularization_type` es `onsite`. Acotado entre `0` y `1` si `regularization_type` es `alpha`. Si se establece en `auto`, se utiliza la regularización automática: el parámetro de regularización óptimo se encuentra mediante búsqueda binaria con la proporción deseada de clasificadores seleccionados respecto al total (`regularization_desired_ratio`) y el límite superior del parámetro de regularización (`regularization_upper_bound`). Si se proporciona una lista, se entrenarán modelos para cada valor y se seleccionará el de mejor rendimiento. | No | `0.01` |
| `regularization_desired_ratio` | `float` o `list` | Proporción(es) deseada(s) de clasificadores seleccionados respecto al total para la regularización automática. Si se proporciona una lista, se entrenarán modelos para cada proporción y se seleccionará el de mejor rendimiento. | No | `0.75` |
| `regularization_upper_bound` | `float` o `list` | Límite(s) superior(es) para el parámetro de regularización al usar la regularización automática. Si se proporciona una lista, se entrenarán modelos para cada límite superior y se seleccionará el de mejor rendimiento. | No | `200` |
| `weight_update_method` | `str` | Método para actualizar los pesos de las muestras entre `logarithmic` y `quadratic`. | No | `logarithmic` |
| `sample_scaling` | `boolean` | Si se debe aplicar escalado de muestras. | No | `False` |
| `prediction_scaling` | `float` | Factor de escala para las predicciones. | No | `None` |
| `optimizer_options` | `dictionary` | Opciones del optimizador QAOA. Una lista de opciones disponibles se presenta más adelante en esta documentación. | No | ... |
| `voting` | `str` | Usar votación por mayoría (`hard`) o promedio de probabilidades (`soft`) para agregar las predicciones/probabilidades de los aprendices. | No | `hard` |
| `prob_threshold` | `float` | Umbral de probabilidad óptimo. | No | `0.5` |
| `random_state` | `integer` | Controla la aleatoriedad para la reproducibilidad. | No | `None` |


- Además, `optimizer_options` se detallan a continuación:

|     Nombre    |    Tipo     | Descripción | Requerido  | Por defecto |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | El número de soluciones | No | `1024` |
| `reps` | `integer` | El número de repeticiones | No | `4` |
| `sparsify` | `float` | El umbral de dispersión | No | `0.001` |
| `theta` | `float` | El valor inicial de theta, un parámetro variacional de QAOA | No | `None` |
| `simulator` | `boolean` | Si se usa un simulador o un QPU | No | `False` |
| `classical_optimizer` | `str` | Nombre del optimizador clásico para el QAOA. Todos los solvers ofrecidos por SciPy, listados [aquí](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), son utilizables. Deberás configurar `classical_optimizer_options` en consecuencia. | No | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opciones del optimizador clásico. Para una lista completa de opciones disponibles, consulta la [documentación de SciPy](https://docs.scipy.org/doc/scipy/reference/) | No | `{"maxiter": 60}` |
| `optimization_level` | `integer` | La profundidad del circuito QAOA | No | `3` |
| `num_transpiler_runs` | `integer` | Número de ejecuciones del transpilador | No | `30` |
| `pass_manager_options` | `dictionary` | Opciones para generar el gestor de pases preestablecido | No | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opciones del Estimator. Para una lista completa de opciones disponibles, consulta la [documentación del cliente Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | No | `None` |
| `sampler_options` | `dictionary` | Opciones del Sampler. Para una lista completa de opciones disponibles, consulta la [documentación del cliente Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | No | `None` |

- Los valores por defecto de `estimator_options` son:

|     Nombre    |    Tipo     | Valor  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Los valores por defecto de `sampler_options` son:

|     Nombre    |    Tipo     | Valor |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Uso

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### Validaciones
- `name`:
    - El nombre debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Ya debe existir un clasificador con el mismo nombre en el directorio de datos compartido.
### 4. Fit
La acción `fit` entrena un clasificador usando los datos de entrenamiento proporcionados.

#### Entradas
|     Nombre    |    Tipo     | Descripción |   Requerido  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | El nombre de la acción. Debe ser `fit`. | Sí |
| `name`      | `str`    | El nombre del clasificador a entrenar. | Sí |
| `X`         | `array` o `list` o `str` | Los datos de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `y`         | `array` o `list` o `str` | Los valores objetivo de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `fit_params`| `dictionary`| Parámetros adicionales para pasar al método `fit` del clasificador. | No |

##### fit_params
|     Nombre    |    Tipo     | Descripción |   Requerido  |   Por defecto   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | Los datos y etiquetas de validación. | No | `None` |
| `pos_label` | `integer` o `str` | La etiqueta de clase que se mapea a 1. | No | `None` |
| `optimization_data` | `str` | Conjunto de datos sobre el que optimizar el ensemble. Puede ser uno de: `train`, `validation`, `both`. | No | `train` |

#### Uso

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### Validaciones
- `name`:
    - El nombre debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Ya debe existir un clasificador con el mismo nombre en el directorio de datos compartido.
### 5. Predict
La acción `predict` se usa para obtener predicciones duras y suaves (probabilidades).

#### Entradas
|     Nombre    |    Tipo     | Descripción |   Requerido  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | El nombre de la acción. Debe ser `predict`. | Sí |
| `name`      | `str`    | El nombre del clasificador a utilizar. | Sí |
| `X`         | `array` o `list` o `str` | Los datos de prueba. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `options["out"]` | `str` | El nombre del archivo JSON de salida para guardar las predicciones en el directorio de datos compartido. Si no se proporciona, las predicciones se devuelven en el resultado del trabajo. | No |

#### Uso

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### Validaciones
- `name`:
    - El nombre debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Ya debe existir un clasificador con el mismo nombre en el directorio de datos compartido.
- `options["out"]`:
    - El nombre del archivo debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Debe tener la extensión `.json`.
### 6. Fit-predict
La acción `fit_predict` entrena un clasificador usando los datos de entrenamiento y luego lo usa para obtener predicciones duras y suaves (probabilidades).

#### Entradas
|     Nombre    |    Tipo     | Descripción |   Requerido  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | El nombre de la acción. Debe ser `fit_predict`. | Sí |
| `name`      | `str`    | El nombre del clasificador a utilizar. | Sí |
| `X_train`   | `array` o `list` o `str` | Los datos de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `y_train`   | `array` o `list` o `str` | Los valores objetivo de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `X_test`    | `array` o `list` o `str` | Los datos de prueba. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `fit_params`| `dictionary`| Parámetros adicionales para pasar al método `fit` del clasificador. | No |
| `options["out"]` | `str` | El nombre del archivo JSON de salida para guardar las predicciones en el directorio de datos compartido. Si no se proporciona, las predicciones se devuelven en el resultado del trabajo. | No |

#### Uso

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### Validaciones
- `name`:
    - El nombre debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Ya debe existir un clasificador con el mismo nombre en el directorio de datos compartido.

- `options["out"]`:
    - El nombre del archivo debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Debe tener la extensión `.json`.
### 7. Create-fit-predict
La acción `create_fit_predict` crea un clasificador, lo entrena usando los datos de entrenamiento proporcionados y luego lo usa para obtener predicciones duras y suaves (probabilidades).

#### Entradas
| Nombre | Tipo | Descripción | Requerido |
|------|------|-------------|-----------|
| `action` | `str` | El nombre de la acción entre `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` y `delete`. | Sí |
| `name` | `str` | El nombre del clasificador a utilizar. | Sí |
| `quantum_classifier` | `str` | El tipo de clasificador, es decir, `QuantumEnhancedEnsembleClassifier`. Por defecto es `QuantumEnhancedEnsembleClassifier`. | No |
| `X_train` | `array` o `list` o `str` | Los datos de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `y_train` | `array` o `list` o `str` | Los valores objetivo de entrenamiento. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `X_test` | `array` o `list` o `str` | Los datos de prueba. Puede ser un array NumPy, una lista o una cadena que referencie un nombre de archivo en el directorio de datos compartido. | Sí |
| `fit_params` | `dictionary` | Parámetros adicionales para pasar al método `fit` del clasificador. | No |
| `options["save"]` | `boolean` | Si se guarda el clasificador entrenado en el directorio de datos compartido. Por defecto es `True`. | No |
| `options["out"]` | `str` | El nombre del archivo JSON de salida para guardar las predicciones en el directorio de datos compartido. Si no se proporciona, las predicciones se devuelven en el resultado del trabajo. | No |

#### Uso

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### Validaciones
- `name`:
    - Si `options["save"]` está establecido en `True`:
        - El nombre debe ser único, una cadena de hasta 64 caracteres.
        - Solo puede incluir caracteres alfanuméricos y guiones bajos.
        - Debe comenzar con una letra y no puede terminar con un guion bajo.
        - No debe existir ya ningún clasificador con el mismo nombre en el directorio de datos compartido.

- `options["out"]`:
    - El nombre del archivo debe ser único, una cadena de hasta 64 caracteres.
    - Solo puede incluir caracteres alfanuméricos y guiones bajos.
    - Debe comenzar con una letra y no puede terminar con un guion bajo.
    - Debe tener la extensión `.json`.
---
## Primeros pasos
Autentícate usando tu [clave de API de IBM Quantum Platform](http://quantum.cloud.ibm.com/) y selecciona la Qiskit Function de la siguiente manera:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Ejemplo
En este ejemplo, usarás la función "Singularity Machine Learning - Clasificación" para clasificar un conjunto de datos que consiste en dos semicírculos en forma de luna entrelazados. El conjunto de datos es sintético, bidimensional y etiquetado con etiquetas binarias. Está diseñado para ser desafiante para algoritmos como la agrupación basada en centroides y la clasificación lineal.
![Conjunto de datos moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
A través de este proceso, aprenderás a crear el clasificador, ajustarlo a los datos de entrenamiento, usarlo para predecir en los datos de prueba y eliminar el clasificador cuando hayas terminado.
Antes de comenzar, necesitas instalar [scikit-learn](https://scikit-learn.org/). Instálalo usando el siguiente comando:
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).